# ⚖️ Legal NLP Model Fine-Tuning
### Fine-tuning `deepset/roberta-base-squad2` (QA) and `facebook/bart-large-cnn` (Summarization) on Indian Legal Judgments

---
**Runtime:** `T4 GPU` (Runtime → Change runtime type → T4 GPU)

**Models to fine-tune:**
- 🔵 `deepset/roberta-base-squad2` → Extractive Question Answering
- 🟠 `facebook/bart-large-cnn` → Abstractive Summarization

**Dataset:** `legal_dataset.json` — 200 Indian Supreme Court & High Court judgments

---
## 🔧 CELL 1 — Environment Setup & GPU Check

In [ ]:
# Verify GPU
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

# Install required libraries
!pip install -q transformers==4.40.0 datasets accelerate evaluate rouge_score sentencepiece

print("\n✅ All packages installed successfully!")

---
## 📤 CELL 2 — Upload Dataset
Upload your `legal_dataset.json` file when prompted.

In [ ]:
from google.colab import files
import json

print("👇 Click 'Choose Files' and upload your legal_dataset.json")
uploaded = files.upload()

filename = list(uploaded.keys())[0]
with open(filename, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

documents = raw_data['documents']
print(f"\n✅ Dataset loaded: {len(documents)} documents")
print(f"📄 First doc ID: {documents[0]['doc_id']}")

---
## 📦 CELL 3 — Imports

In [ ]:
import os
import json
import torch
import warnings
warnings.filterwarnings('ignore')

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import evaluate

# Device info
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"🖥️  Using device: {device}")
if device == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
# 🔵 PART A — Fine-Tune `deepset/roberta-base-squad2` for Legal QA

## CELL 4A — Parse QA Dataset

In [ ]:
qa_records = []
skipped = 0

for doc in documents:
    context = doc['judgment']['judgment_text']
    for qa in doc.get('qa_pairs', []):
        question = qa['question']
        answer_text = qa['answer']

        # Find exact span in context
        start = context.find(answer_text)
        if start == -1:
            # Try case-insensitive match
            ctx_lower = context.lower()
            ans_lower = answer_text.lower()
            start = ctx_lower.find(ans_lower)
            if start != -1:
                answer_text = context[start:start + len(answer_text)]

        if start == -1:
            skipped += 1
            continue  # Skip paraphrased answers not in context

        qa_records.append({
            'id': f"{doc['doc_id']}_{len(qa_records)}",
            'context': context,
            'question': question,
            'answers': {
                'text': [answer_text],
                'answer_start': [start]
            }
        })

print(f"✅ QA pairs collected : {len(qa_records)}")
print(f"⚠️  QA pairs skipped  : {skipped} (paraphrased answers not in context)")

# Train / Validation split (90/10)
split_idx = int(len(qa_records) * 0.9)
qa_dataset = DatasetDict({
    'train': Dataset.from_list(qa_records[:split_idx]),
    'validation': Dataset.from_list(qa_records[split_idx:])
})
print(f"\n📊 Train: {len(qa_dataset['train'])} | Validation: {len(qa_dataset['validation'])}")

## CELL 5A — Tokenize QA Dataset

In [ ]:
QA_MODEL_NAME = "deepset/roberta-base-squad2"
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)

MAX_LENGTH = 512
DOC_STRIDE = 128

def preprocess_qa(examples):
    tokenized = qa_tokenizer(
        examples['question'],
        examples['context'],
        truncation='only_second',
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding='max_length'
    )

    sample_mapping = tokenized.pop('overflow_to_sample_mapping')
    offset_mapping = tokenized.pop('offset_mapping')

    start_positions = []
    end_positions = []

    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_mapping[i]
        answers = examples['answers'][sample_idx]
        answer_start_char = answers['answer_start'][0]
        answer_end_char = answer_start_char + len(answers['text'][0])

        # Find token positions
        sequence_ids = tokenized.sequence_ids(i)
        # Context tokens are where sequence_id == 1
        ctx_start = next((j for j, s in enumerate(sequence_ids) if s == 1), 0)
        ctx_end = len(sequence_ids) - 1 - next((j for j, s in enumerate(reversed(sequence_ids)) if s == 1), 0)

        token_start_idx, token_end_idx = ctx_start, ctx_start
        found = False
        for j in range(ctx_start, ctx_end + 1):
            if offsets[j][0] <= answer_start_char < offsets[j][1]:
                token_start_idx = j
                found = True
            if offsets[j][0] < answer_end_char <= offsets[j][1]:
                token_end_idx = j
                break

        if not found:
            token_start_idx, token_end_idx = 0, 0

        start_positions.append(token_start_idx)
        end_positions.append(token_end_idx)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions'] = end_positions
    return tokenized

tokenized_qa = qa_dataset.map(
    preprocess_qa,
    batched=True,
    remove_columns=qa_dataset['train'].column_names
)

print(f"✅ Tokenization complete!")
print(f"   Train features: {len(tokenized_qa['train'])}")
print(f"   Val features  : {len(tokenized_qa['validation'])}")

## CELL 6A — Train RoBERTa QA Model

In [ ]:
qa_model = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)

qa_training_args = TrainingArguments(
    output_dir='./roberta-legal-qa',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    warmup_steps=50,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    learning_rate=2e-5,
    fp16=True,
    logging_steps=10,
    logging_dir='./logs-qa',
    report_to='none',
    dataloader_num_workers=2,
)

qa_trainer = Trainer(
    model=qa_model,
    args=qa_training_args,
    train_dataset=tokenized_qa['train'],
    eval_dataset=tokenized_qa['validation'],
    tokenizer=qa_tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Starting RoBERTa QA fine-tuning...")
qa_trainer.train()
print("\n✅ RoBERTa QA training complete!")

## CELL 7A — Evaluate & Test RoBERTa QA

In [ ]:
from transformers import pipeline

# Save model
qa_trainer.save_model('./roberta-legal-qa-final')
qa_tokenizer.save_pretrained('./roberta-legal-qa-final')
print("✅ Model saved to ./roberta-legal-qa-final")

# Quick inference test
qa_pipe = pipeline(
    'question-answering',
    model='./roberta-legal-qa-final',
    tokenizer='./roberta-legal-qa-final',
    device=0 if device == 'cuda' else -1
)

# Test with the first document
test_doc = documents[0]
test_context = test_doc['judgment']['judgment_text']
test_questions = [
    "Which court delivered this judgment?",
    "What was the final outcome of the case?",
    "Which constitutional article was primarily violated?"
]

print("\n🧪 Inference Test on DOC_0001 (Electoral Bonds Case):")
print("-" * 60)
for q in test_questions:
    result = qa_pipe(question=q, context=test_context)
    print(f"Q: {q}")
    print(f"A: {result['answer']} (score: {result['score']:.3f})")
    print()

---
# 🟠 PART B — Fine-Tune `facebook/bart-large-cnn` for Legal Summarization

## CELL 4B — Parse Summarization Dataset

In [ ]:
sum_records = []

for doc in documents:
    judgment_text = doc['judgment'].get('judgment_text', '').strip()
    summary = doc['judgment'].get('summary', '').strip()
    if judgment_text and summary and len(judgment_text) > 100:
        sum_records.append({
            'document': judgment_text,
            'summary': summary
        })

print(f"✅ Summarization records: {len(sum_records)}")

# Train / Validation split (90/10)
split_idx = int(len(sum_records) * 0.9)
sum_dataset = DatasetDict({
    'train': Dataset.from_list(sum_records[:split_idx]),
    'validation': Dataset.from_list(sum_records[split_idx:])
})
print(f"📊 Train: {len(sum_dataset['train'])} | Validation: {len(sum_dataset['validation'])}")

## CELL 5B — Tokenize Summarization Dataset

In [ ]:
BART_MODEL_NAME = "facebook/bart-large-cnn"
bart_tokenizer = AutoTokenizer.from_pretrained(BART_MODEL_NAME)

MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 256

def preprocess_summarization(examples):
    # Tokenize inputs
    model_inputs = bart_tokenizer(
        examples['document'],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding='max_length'
    )
    # Tokenize targets (summaries)
    labels = bart_tokenizer(
        text_target=examples['summary'],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding='max_length'
    )
    # Replace padding token id with -100 so it's ignored in loss
    labels['input_ids'] = [
        [(l if l != bart_tokenizer.pad_token_id else -100) for l in label]
        for label in labels['input_ids']
    ]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_sum = sum_dataset.map(
    preprocess_summarization,
    batched=True,
    remove_columns=['document', 'summary']
)

print("✅ Summarization tokenization complete!")
print(f"   Train features: {len(tokenized_sum['train'])}")
print(f"   Val features  : {len(tokenized_sum['validation'])}")

## CELL 6B — Train BART Summarization Model

In [ ]:
bart_model = AutoModelForSeq2SeqLM.from_pretrained(BART_MODEL_NAME)
data_collator = DataCollatorForSeq2Seq(bart_tokenizer, model=bart_model, pad_to_multiple_of=8)

# ROUGE metric for evaluation
rouge = evaluate.load('rouge')

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = bart_tokenizer.batch_decode(predictions, skip_special_tokens=True)
    # Replace -100 in labels
    labels = [[l for l in label if l != -100] for label in labels]
    decoded_labels = bart_tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )
    return {k: round(v, 4) for k, v in result.items()}

bart_training_args = Seq2SeqTrainingArguments(
    output_dir='./bart-legal-summarizer',
    num_train_epochs=4,
    per_device_train_batch_size=2,          # BART-large is memory-heavy
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,          # Effective batch = 16
    warmup_steps=30,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='rouge2',
    predict_with_generate=True,
    generation_max_length=256,
    fp16=True,
    learning_rate=5e-5,
    logging_steps=5,
    logging_dir='./logs-bart',
    report_to='none',
    dataloader_num_workers=2,
)

bart_trainer = Seq2SeqTrainer(
    model=bart_model,
    args=bart_training_args,
    train_dataset=tokenized_sum['train'],
    eval_dataset=tokenized_sum['validation'],
    tokenizer=bart_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Starting BART Summarization fine-tuning...")
print("⏱️  Expected time on T4: ~25-40 minutes")
bart_trainer.train()
print("\n✅ BART Summarization training complete!")

## CELL 7B — Evaluate & Test BART Summarizer

In [ ]:
# Save model
bart_trainer.save_model('./bart-legal-summarizer-final')
bart_tokenizer.save_pretrained('./bart-legal-summarizer-final')
print("✅ BART model saved to ./bart-legal-summarizer-final")

# Inference test
sum_pipe = pipeline(
    'summarization',
    model='./bart-legal-summarizer-final',
    tokenizer='./bart-legal-summarizer-final',
    device=0 if device == 'cuda' else -1
)

# Test on first document
test_text = documents[0]['judgment']['judgment_text']
gold_summary = documents[0]['judgment']['summary']

generated = sum_pipe(
    test_text,
    max_length=256,
    min_length=80,
    num_beams=4,
    early_stopping=True
)[0]['summary_text']

print("\n🧪 Inference Test on DOC_0001 (Electoral Bonds Case):")
print("-" * 60)
print("📝 GENERATED SUMMARY:")
print(generated)
print("\n📚 GOLD SUMMARY (Reference):")
print(gold_summary)

# ROUGE score on test sample
score = rouge.compute(predictions=[generated], references=[gold_summary])
print(f"\n📊 ROUGE Scores:")
for k, v in score.items():
    print(f"   {k}: {v:.4f}")

---
## 📦 CELL 8 — Package & Download Both Models
This creates `.tar.gz` archives you can download and deploy to GCP/AWS.

In [ ]:
import os
import tarfile
from google.colab import files

def create_archive(src_dir, archive_name):
    print(f"📦 Packaging {src_dir} → {archive_name}...")
    with tarfile.open(archive_name, 'w:gz') as tar:
        tar.add(src_dir, arcname=os.path.basename(src_dir))
    size_mb = os.path.getsize(archive_name) / (1024 * 1024)
    print(f"   ✅ Archive size: {size_mb:.1f} MB")
    return archive_name

# Package models
qa_archive    = create_archive('./roberta-legal-qa-final',       'roberta_legal_qa.tar.gz')
bart_archive  = create_archive('./bart-legal-summarizer-final',  'bart_legal_summarizer.tar.gz')

print("\n📥 Downloading models to your local machine...")
files.download(qa_archive)
files.download(bart_archive)

print("\n✅ Both models downloaded!")
print("📌 Next step: Follow the deployment guide to host these on GCP/AWS")

---
## ☁️ CELL 9 (Optional) — Push Models to Hugging Face Hub
Instead of downloading locally, you can push directly to HF Hub for easier cloud deployment.

In [ ]:
# --------------------------------------------------
# OPTIONAL: Push to Hugging Face Hub
# 1. Create a free account at https://huggingface.co
# 2. Get your token from: https://huggingface.co/settings/tokens
# --------------------------------------------------

from huggingface_hub import login

HF_TOKEN = ""  # ← Paste your HF token here
HF_USERNAME = ""  # ← Your HuggingFace username

if HF_TOKEN and HF_USERNAME:
    login(token=HF_TOKEN)

    # Push RoBERTa QA model
    qa_trainer.push_to_hub(f"{HF_USERNAME}/roberta-legal-qa")
    print(f"✅ QA model pushed → hf.co/{HF_USERNAME}/roberta-legal-qa")

    # Push BART summarizer
    bart_trainer.push_to_hub(f"{HF_USERNAME}/bart-legal-summarizer")
    print(f"✅ BART model pushed → hf.co/{HF_USERNAME}/bart-legal-summarizer")
else:
    print("⚠️  Skipped: Fill in HF_TOKEN and HF_USERNAME above to enable.")

---
## 📊 CELL 10 — Final Training Summary

In [ ]:
print("=" * 60)
print("         FINE-TUNING COMPLETE — SUMMARY")
print("=" * 60)

print("\n🔵 RoBERTa QA Model")
print(f"   Base model : deepset/roberta-base-squad2")
print(f"   Saved to   : ./roberta-legal-qa-final/")

qa_log = qa_trainer.state.log_history
qa_eval = [l for l in qa_log if 'eval_loss' in l]
if qa_eval:
    best = min(qa_eval, key=lambda x: x['eval_loss'])
    print(f"   Best eval loss: {best['eval_loss']:.4f} (epoch {best['epoch']:.0f})")

print("\n🟠 BART Summarization Model")
print(f"   Base model : facebook/bart-large-cnn")
print(f"   Saved to   : ./bart-legal-summarizer-final/")

bart_log = bart_trainer.state.log_history
bart_eval = [l for l in bart_log if 'eval_rouge2' in l]
if bart_eval:
    best_bart = max(bart_eval, key=lambda x: x['eval_rouge2'])
    print(f"   Best ROUGE-2: {best_bart['eval_rouge2']:.4f} (epoch {best_bart['epoch']:.0f})")

print("\n📌 Files ready for deployment:")
print("   roberta_legal_qa.tar.gz")
print("   bart_legal_summarizer.tar.gz")
print("\n✅ Follow deployment guide to host on GCP/AWS!")
print("=" * 60)